### 🔁 What is Iterative Retrieval in Agentic RAG?
Combined both Iterative And Self reflection

✅ Definition:
Iterative Retrieval is a dynamic strategy where an AI agent doesn't settle for the first batch of retrieved documents. Instead, it evaluates the adequacy of the initial context, and if necessary, it:

- Refines the query,
- Retrieves again,
- Repeats the process until it’s confident enough to answer the original question.

🧠 Why Use It?
In standard RAG:

- A single retrieval step is done, and the LLM uses it to answer.
- If the documents were incomplete or irrelevant, the answer may fail.

In Iterative RAG:

- The agent reflects on the retrieved content and the answer it produced.
- If it’s unsure, it can refine its search (like a human researcher would).

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# LLM Model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
llm

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x00000233923078C0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000233927A0440>, root_client=<openai.OpenAI object at 0x00000233923056A0>, root_async_client=<openai.AsyncOpenAI object at 0x00000233927A01A0>, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# -------------------------------
# 1. Prepare Vectorstore
# -------------------------------

# Laod data
loader = TextLoader("research.txt", encoding="utf-8")
docs = loader.load()

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# perform embeddings
embedding = OpenAIEmbeddings()

# create vectorstore
vectorstore = FAISS.from_documents(chunks, embedding)

# create retriever
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002339285F4D0>, search_kwargs={})

In [3]:
from typing import List
from pydantic import BaseModel
from langchain.schema import Document

# -------------------------------
# 2. LangGraph State Definition
# -------------------------------

class IterativeRAGState(BaseModel):
    question: str
    refined_question: str = ""
    retrieved_docs: List[Document] = []
    answer: str = ""
    verified: bool = False
    attempts: int = 0

In [4]:
# -------------------------------
# 3. Nodes
# -------------------------------

# a. Retrieve docs
def retrieve_docs(state: IterativeRAGState) -> IterativeRAGState:
    query = state.refined_question or state.question
    docs = retriever.invoke(query)
    return state.model_copy(update={"retrieved_docs": docs})


# b. Reflect And Verify
def generate_answer(state: IterativeRAGState) -> IterativeRAGState:
    
    context = "\n\n".join(doc.page_content for doc in state.retrieved_docs)
    prompt = f"""Use the following context to answer the question:

Context: {context}

Question: {state.question}
"""

    response = llm.invoke(prompt.strip()).content.strip()
    return state.model_copy(update={"answer": response, "attempts": state.attempts + 1})


# c. Reflect on answer
def reflect_on_answer(state: IterativeRAGState) -> IterativeRAGState:
    
    prompt = f"""
Evaluate whether the answer below is factually sufficient and complete.

Question: {state.question}
Answer: {state.answer}

Respond 'YES' if it's complete, otherwise 'NO' with feedback.
"""

    feedback = llm.invoke(prompt).content.lower()
    verified = "yes" in feedback
    return state.model_copy(update={"verified": verified})


# d. Refine query
def refine_query(state: IterativeRAGState) -> IterativeRAGState:
    
    prompt = f"""
The answer appears incomplete. Suggest a better version of the query that would help retrieve more relevant context.

Original Question: {state.question}
Current Answer: {state.answer}
"""

    new_query = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"refined_question": new_query})


In [8]:
from langgraph.graph import StateGraph,START, END
# -------------------------------
# 4. LangGraph Graph
# -------------------------------

# Build the graph
graph = StateGraph(IterativeRAGState)

# Nodes
graph.add_node("retrieve", retrieve_docs)
graph.add_node("answer", generate_answer)
graph.add_node("reflect", reflect_on_answer)
graph.add_node("refine", refine_query)

# Edges
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "answer")
graph.add_edge("answer", "reflect")

graph.add_conditional_edges(
    "reflect",
    lambda s: END if s.verified or s.attempts >= 2 else "refine"
)

graph.add_edge("refine", "retrieve")
graph.add_edge("answer", END)

# Compile
workflow = graph.compile()

In [ ]:
# -------------------------------
# 5. Run  RAG Agent
# -------------------------------

query = "agent loops  and transformer-based systems?"

initial_state = IterativeRAGState(question=query)
final = workflow.invoke(initial_state)

print("✅ Final Answer:\n", final["answer"])
print("\n🧠 Verified:", final["verified"])
print("🔁 Attempts:", final["attempts"])


✅ Final Answer:
 Agent loops are often used in conjunction with transformer-based systems in order to enable interaction and decision-making within an automated system. This allows the system to receive input, process it using the transformer-based model, and produce an output or action based on the input. The transformer-based system provides the ability to understand and generate text, while the agent loop manages the overall flow and decision-making process of the system. In the context of the experiments and evaluations described, agent loops play a crucial role in utilizing transformer models effectively for tasks such as support ticket tagging, chatbot Q&A, and efficient image classification.

🧠 Verified: True
🔁 Attempts: 1
